In [ ]:
# bowaka_v2_lab notebook bootstrap cell — DO NOT EDIT BY HAND.
# Adds the lab's src/ (and its bowaka_common dependency) to sys.path and pins
# the working directory to the repo root, so `import bowaka_v2_lab` and
# repo-root-relative CONFIG_PATH parameters resolve identically under jupyter,
# papermill, and the QuantsLab scheduler.
import os
import sys
from pathlib import Path

_lab_root = None
for _candidate in [Path.cwd(), *Path.cwd().parents]:
    if (_candidate / "src" / "bowaka_v2_lab" / "__init__.py").is_file():
        _lab_root = _candidate
        break
if _lab_root is None:
    raise RuntimeError(
        f"bowaka_v2_lab bootstrap: src/bowaka_v2_lab/ not found at or above {Path.cwd()}"
    )

# Pin CWD to the repo root (the directory holding research_notebooks/ and the
# Makefile) so repo-root-relative CONFIG_PATH values resolve regardless of how
# the notebook was launched (jupyter CWD = notebook dir, scheduler = repo root).
_repo_root = _lab_root
for _candidate in [_lab_root, *_lab_root.parents]:
    if (_candidate / "research_notebooks").is_dir() and (_candidate / "Makefile").is_file():
        _repo_root = _candidate
        break
os.chdir(_repo_root)

# Make the lab and its bowaka_common dependency importable from the working
# tree, even when the packages are not pip-installed. v1 bowaka_lab is
# deliberately excluded — v2 must not import v1.
for _src in (_lab_root / "src",
             _repo_root / "research_notebooks" / "bowaka_common" / "src"):
    if _src.is_dir() and str(_src) not in sys.path:
        sys.path.insert(0, str(_src))

import bowaka_v2_lab  # noqa: F401
print(f"bowaka_v2_lab {bowaka_v2_lab.__version__} (cwd={_repo_root})")


In [ ]:
# Papermill parameter cell.
CONFIG_PATH = 'research_notebooks/bowaka_v2_lab/configs/bowaka_v2_backtest_smoke.yml'


# 02 — Universe Backfill & PIT Snapshot

Builds a point-in-time universe snapshot. Symbols and price/ADV baselines come from the shared lake (research config) or are synthetic (smoke config).

In [ ]:
import pandas as pd
from bowaka_v2_lab.config import load_config
from bowaka_v2_lab.backtest_runner import resolve_symbols, config_sessions, uses_lake
from bowaka_v2_lab.scanner.universe_builder import build_universe_snapshot
cfg = load_config(CONFIG_PATH)
md = cfg.get('market_data', {})
session = config_sessions(cfg)[-1]
syms = resolve_symbols(cfg)
cols = ['symbol', 'prior_close', 'avg_dollar_volume_20d']
if uses_lake(cfg):
    from bowaka_v2_lab.data.suppliers import build_daily_cache_from_lake
    cache = build_daily_cache_from_lake(md.get('shared_root'), syms, session, feed=md.get('feed', 'iex'))
    baselines = cache[cols] if not cache.empty else pd.DataFrame(columns=cols)
else:
    baselines = pd.DataFrame([{'symbol': s, 'prior_close': 100.0,
                               'avg_dollar_volume_20d': 5_000_000} for s in syms])
asset_master = pd.DataFrame([{'symbol': s, 'exchange': 'NASDAQ', 'venue_code': 'XNAS',
  'instrument_class': 'operating_equity', 'eligible_for_bowaka_equity_bucket': True}
  for s in syms])
snap = build_universe_snapshot(asset_master=asset_master, daily_baselines=baselines,
  cfg=cfg, session_date=session)
n_pass = len(snap['symbols'])
print('session', session, '-', n_pass, 'symbols pass the universe gate',
      'of', len(syms), 'candidates')
